# Bundle Migration: Export & Transform (Modular)

## Overview

Export and transform dashboards using centralized configuration and helper modules.

**What this notebook does:**
1. Loads config from `config/config.yaml`
2. Discovers dashboards from source workspace
3. Exports dashboard JSONs and permissions
4. Applies catalog/schema transformations
5. Saves transformed files for bundle generation

**Next step:** Run `Bundle_02_Generate_and_Deploy.ipynb`

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
# VERIFY APPROVED INVENTORY EXISTS AND IS RECENT

import datetime

print("="*70)
print("STEP 2: EXPORT & TRANSFORM")
print("="*70)
print("\n🔍 Verifying approved inventory...\n")

# Get configuration
volume_base = dbutils.widgets.get("volume_base")
inventory_path = dbutils.widgets.get("inventory_path")  # Should be dashboard_inventory_approved
approved_csv_path = f"{volume_base}/{inventory_path}/inventory.csv"

# Check if file exists
try:
    file_info = dbutils.fs.ls(f"{volume_base}/{inventory_path}/")
    csv_found = False
    
    for file in file_info:
        if file.name == 'inventory.csv':
            csv_found = True
            modified_time = datetime.datetime.fromtimestamp(file.modificationTime / 1000)
            age_hours = (datetime.datetime.now() - modified_time).total_seconds() / 3600
            
            print(f"✅ Approved inventory found")
            print(f"   Location: {approved_csv_path}")
            print(f"   Modified: {modified_time.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Age: {age_hours:.1f} hours ago")
            
            # Warn if file is old (older than 7 days)
            if age_hours > 168:
                print(f"\n⚠️  WARNING: File is {age_hours/24:.1f} days old")
                print(f"   Consider regenerating inventory if dashboards changed")
            
            # Load and show count
            df_approved = spark.read.csv(approved_csv_path, header=True, inferSchema=True)
            print(f"   Dashboards: {df_approved.count()}\n")
            break
    
    if not csv_found:
        raise FileNotFoundError("inventory.csv not found")
        
except Exception as e:
    print("="*70)
    print("❌ ERROR: APPROVED INVENTORY NOT FOUND")
    print("="*70)
    print(f"\nExpected location: {approved_csv_path}")
    print("\n🎯 REQUIRED ACTIONS:")
    print("   1. Run Step 1 if you haven't: databricks bundle run inventory_generation -t dev")
    print("   2. Then either:")
    print("      Option A: Manually edit dashboard_inventory/inventory.csv")
    print("                and upload to dashboard_inventory_approved/inventory.csv")
    print("      Option B: Run helper notebook:")
    print("                Bundle/Bundle_00a_Review_and_Approve_Inventory.ipynb")
    print(f"\nError details: {e}")
    print("="*70)
    raise FileNotFoundError(f"Approved inventory not found at {approved_csv_path}")

print("="*70)
print("✅ VERIFICATION PASSED - STARTING EXPORT")
print("="*70 + "\n")

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

# Add helpers to path
sys.path.insert(0, '../helpers')

from helpers import (
    load_config,
    get_path,
    create_workspace_client,
    discover_dashboards,
    export_dashboard,
    get_dashboard_permissions,
    load_mapping_csv,
    transform_dashboard_json,
    write_volume_file,
    ensure_directory_exists
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration
config = load_config('../config/config.yaml')

print("\n✅ Configuration loaded\n")
print(f"   Source: {config['source']['workspace_url']}")
print(f"   Volume base: {config['paths']['volume_base']}")
print(f"   Discovery method: {config['dashboard_selection']['method']}")

# Ensure directories exist
export_path = get_path('exported')
transformed_path = get_path('transformed')

print("\n📁 Ensuring directories exist...")
ensure_directory_exists(export_path)
ensure_directory_exists(transformed_path)
print(f"   ✅ Export: {export_path}")
print(f"   ✅ Transformed: {transformed_path}")

## Cell 3: Discover Dashboards

In [ ]:
print("\n" + "="*80)
print("STEP 1: GENERATE INVENTORY")
print("="*80)

# Connect to source workspace
print("\n🔗 Connecting to source workspace...")
source_client = create_workspace_client('source')
print(f"   ✅ Connected\n")

# Generate comprehensive inventory with metadata
print(f"🔍 Generating inventory using: {config['dashboard_selection']['method']}")

from helpers import generate_inventory, save_inventory_to_csv

inventory = generate_inventory(
    source_client,
    include_published_status=True,
    include_metadata=True
)

print(f"\n✅ Generated inventory: {len(inventory)} dashboard(s)\n")

if not inventory:
    print("⚠️  No dashboards found. Check your configuration:")
    print(f"   Method: {config['dashboard_selection']['method']}")
    if config['dashboard_selection']['method'] == 'catalog_filter':
        print(f"   Catalog: {config['dashboard_selection']['catalog_filter']['catalog']}")

## Cell 3a: Save Inventory to CSV

In [ ]:
if inventory:
    # Save inventory to CSV for review and record-keeping
    inventory_csv_path = f"{get_path('volume_base')}/dashboard_inventory/inventory.csv"
    
    print("\n📝 Saving inventory to CSV...")
    save_inventory_to_csv(inventory, inventory_csv_path)
    
    print(f"\n✅ Inventory saved to: {inventory_csv_path}")
    print("\n💡 You can:")
    print("   - Download and edit this CSV")
    print("   - Remove dashboards you don't want to migrate")
    print("   - Add notes or tags")
    print("   - Use it for approval workflow")
else:
    print("\n⚠️  No inventory to save")

## Cell 3b: Display Inventory for Review

**REVIEW THE INVENTORY BELOW:**

- Check if all expected dashboards are listed
- Note which ones are published vs not published
- Identify any test/temporary dashboards to exclude

In [ ]:
if inventory:
    print("="*80)
    print("DASHBOARD INVENTORY - REVIEW THIS LIST")
    print("="*80)
    
    # Display as formatted table
    df = pd.DataFrame(inventory)
    
    # Reorder columns for readability
    cols_order = ['dashboard_name', 'published', 'dashboard_id', 'path', 'warehouse_id']
    cols_present = [c for c in cols_order if c in df.columns]
    display(df[cols_present])
    
    # Show summary
    print(f"\n📊 Summary:")
    print(f"   Total dashboards: {len(inventory)}")
    if 'published' in df.columns:
        pub_count = len(df[df['published'] == 'Yes'])
        unpub_count = len(df[df['published'] == 'No'])
        print(f"   Published: {pub_count}")
        print(f"   Unpublished: {unpub_count}")
    
    print("\n" + "="*80)
    print("⚠️  REVIEW THE INVENTORY ABOVE")
    print("="*80)
    print("\n📝 To exclude dashboards from migration:")
    print("   1. Open the CSV file in the volume")
    print(f"      Path: {inventory_csv_path}")
    print("   2. Delete rows for dashboards you don't want to migrate")
    print("   3. Save the file")
    print("   4. Re-run the next cell")
    print("\n✅ If inventory looks good, proceed to next cell")
else:
    print("\n⚠️  No inventory to display")

## Cell 3c: Confirm and Load Validated Inventory

**MANUAL CHECKPOINT:** Run this cell only after reviewing the inventory above.

If you need to edit the inventory, edit the CSV file in the volume and re-run this cell.

In [ ]:
print("="*80)
print("STEP 2: LOAD VALIDATED INVENTORY")
print("="*80)

# Confirmation flag - set to True when you've reviewed the inventory
INVENTORY_CONFIRMED = False  # ← Change to True after review

if not INVENTORY_CONFIRMED:
    print("\n⚠️  INVENTORY NOT CONFIRMED")
    print("\n📋 Please:")
    print("   1. Review the inventory displayed above")
    print("   2. Edit CSV if needed (remove unwanted dashboards)")
    print("   3. Set INVENTORY_CONFIRMED = True in this cell")
    print("   4. Re-run this cell")
    print("\n❌ Migration will NOT proceed until confirmed")
    
else:
    # Load the validated inventory from CSV
    print("\n✅ Inventory confirmed. Loading validated list...")
    
    from helpers import load_inventory_from_csv
    
    inventory_csv_path = f"{get_path('volume_base')}/dashboard_inventory/inventory.csv"
    dashboards = load_inventory_from_csv(inventory_csv_path)
    
    print(f"\n✅ Loaded {len(dashboards)} validated dashboard(s)")
    print("\n📊 Dashboard list:")
    for i, dash in enumerate(dashboards, 1):
        pub_status = dash.get('published', 'Unknown')
        print(f"   {i}. {dash['dashboard_name']} (Published: {pub_status})")
    
    print("\n▶️  Ready to proceed with export and transformation")

## Cell 4: Export Dashboards & Permissions (Only After Confirmation)

In [ ]:
print("\n" + "="*80)
print("STEP 3: EXPORTING DASHBOARDS & PERMISSIONS")
print("="*80)

# Check if inventory was confirmed
if 'INVENTORY_CONFIRMED' not in dir() or not INVENTORY_CONFIRMED:
    print("\n❌ CANNOT PROCEED: Inventory not confirmed")
    print("\n⚠️  Please run the previous cell and set INVENTORY_CONFIRMED = True")
    raise Exception("Inventory must be confirmed before export")

if not dashboards:
    print("\n⚠️  No dashboards to export")
else:
    export_results = []
    
    for i, dash in enumerate(dashboards, 1):
        dashboard_id = dash.get('dashboard_id') or dash.get('id')
        dashboard_name = dash.get('dashboard_name') or dash.get('name')
        print(f"\n[{i}/{len(dashboards)}] Exporting: {dashboard_name}")
        
        try:
            # Export dashboard JSON
            json_content, display_name, clean_name = export_dashboard(source_client, dashboard_id)
            
            # Save JSON
            json_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
            write_volume_file(json_file, json_content)
            print(f"   ✅ Exported JSON")
            
            # Get and save permissions
            permissions = get_dashboard_permissions(source_client, dashboard_id)
            permissions['display_name'] = display_name
            
            perm_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
            write_volume_file(perm_file, json.dumps(permissions, indent=2))
            
            acl_count = len(permissions.get('access_control_list', []))
            print(f"   🔐 Permissions: {acl_count} ACL(s)")
            
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': display_name,
                'status': 'success',
                'json_file': json_file
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': dash['name'],
                'status': 'failed'
            })
    
    successful = len([r for r in export_results if r['status'] == 'success'])
    print(f"\n✅ Successfully exported: {successful}/{len(dashboards)}")

## Cell 5: Transform Dashboards (Catalog/Schema Remapping)

In [ ]:
print("\n" + "="*80)
print("TRANSFORMING DASHBOARDS")
print("="*80)

if not config['transformation']['enabled']:
    print("\n⚠️  Transformation disabled in config")
elif not export_results or successful == 0:
    print("\n⚠️  No dashboards to transform")
else:
    # Load mapping CSV
    mapping_csv_path = get_path('volume_base') + "/" + config['transformation']['mapping_csv']
    print(f"\n📋 Loading mappings: {mapping_csv_path}")
    
    try:
        mappings = load_mapping_csv(mapping_csv_path)
        print(f"   ✅ Loaded {len(mappings)} mapping rule(s)\n")
        
    except Exception as e:
        print(f"   ❌ Failed to load CSV: {e}")
        raise
    
    # Transform each dashboard
    transform_results = []
    successful_exports = [r for r in export_results if r['status'] == 'success']
    
    for i, result in enumerate(successful_exports, 1):
        name = result['name']
        json_file = result['json_file']
        
        print(f"[{i}/{len(successful_exports)}] Transforming: {name}")
        
        try:
            # Read exported JSON
            from helpers.volume_utils import read_volume_file
            json_content = read_volume_file(json_file)
            
            # Apply transformations
            transformed = transform_dashboard_json(json_content, mappings)
            
            # Save transformed version
            filename = Path(json_file).name
            transformed_file = f"{transformed_path}/{filename}"
            write_volume_file(transformed_file, transformed)
            
            print(f"   ✅ Transformed")
            
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'success'
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'failed'
            })
    
    successful_transforms = len([r for r in transform_results if r['status'] == 'success'])
    
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"\n✅ Exported: {successful}/{len(dashboards)}")
    print(f"✅ Transformed: {successful_transforms}/{successful}")
    print(f"\n📁 Files ready at: {transformed_path}")
    print(f"\n▶️  Next: Run Bundle_02_Generate_and_Deploy.ipynb")